# Config 

> Configuration options and global defaults

In [ ]:
#| default_exp utils.config

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:

#| export
from __future__ import annotations
from typing import Dict, Optional, Any
import os, json
from pathlib import Path
from types import SimpleNamespace

## Config object
Dot-accessible, mutable config inspired by scanpy.cfg; remains optional thanks to helper functions.

In [ ]:
#| export
class DotConfig(dict):
    '''Dot-accessible configuration with dictionary semantics.'''
    def __getattr__(self, key: str):
        try:
            return self[key]
        except KeyError as exc:
            raise AttributeError(key) from exc

    def __setattr__(self, key: str, value: Any) -> None:
        self[key] = value

    def copy(self) -> "DotConfig":
        '''Return a shallow copy as another DotConfig.'''
        return DotConfig(self)

    def update_from_env(self) -> None:
        '''Refresh config from environment variables if present.'''
        self.log_level = os.environ.get("NBMCP_LOG_LEVEL", self.log_level)
        self.prompt_dir = os.environ.get("NBMCP_PROMPT_DIR", self.prompt_dir)
        self.env_dir_name = os.environ.get("NBMCP_ENV_DIR", self.env_dir_name)


DEFAULT_CONFIG = DotConfig({
    "log_level": os.environ.get("NBMCP_LOG_LEVEL", "INFO"),
    "prompt_dir": os.environ.get("NBMCP_PROMPT_DIR", "prompt_templates"),
    "env_dir_name": os.environ.get("NBMCP_ENV_DIR", "Environments"),
    "max_tree_files": 600,
})
'''Default settings used by nbdev-mcp. Environment variables can override these defaults.'''

NBMCP_CONFIG = DEFAULT_CONFIG
'''Singleton configuration shared across the package. Dot-accessible (``NBMCP_CONFIG.log_level``).'''

CURRENT_PROJECT: Optional[Path] = None
'''Currently selected nbdev project; ``None`` until explicitly set.'''

In [ ]:
#| export
def configure(**kwargs: Any) -> DotConfig:
    '''Update the global NBMCP_CONFIG in-place and return it.'''
    NBMCP_CONFIG.update(kwargs)
    return NBMCP_CONFIG


def get_config() -> DotConfig:
    '''Return the global config object (mutable, dot-accessible).'''
    return NBMCP_CONFIG


def get_current_project() -> Optional[Path]:
    '''Return the currently active project path (if any).'''
    return CURRENT_PROJECT


def set_current_project(path: Optional[Path]) -> None:
    '''Set the active project path used by tools/resources.'''
    global CURRENT_PROJECT
    CURRENT_PROJECT = path


## Config & Bookmarks

In [ ]:
#| export
def find_config_dir() -> Path:
    '''Return the directory for storing MCP nbdev config (bookmarks).

    On macOS/Linux it defaults to ``~/.config/mcp.nbdev``; on Windows it uses
    ``%APPDATA%/mcp.nbdev``. The directory is created if missing.
    '''
    if os.name == "nt":
        base = Path(os.environ.get("APPDATA", Path.home() / "AppData" / "Roaming"))
    else:
        base = Path(os.environ.get("XDG_CONFIG_HOME", Path.home() / ".config"))
    d = base / "mcp.nbdev"
    d.mkdir(parents=True, exist_ok=True)
    return d


BOOKMARKS_PATH: Path = find_config_dir() / "projects.json"
'''Path to the bookmarks JSON file for nbdev-mcp.'''


def load_bookmarks() -> Dict[str, str]:
    '''Load saved project aliases from BOOKMARKS_PATH.'''
    if BOOKMARKS_PATH.exists():
        try:
            data = json.loads(BOOKMARKS_PATH.read_text(encoding="utf-8"))
            aliases = data.get("aliases", {}) if isinstance(data, dict) else {}
            return {k: str(Path(v)) for k, v in aliases.items()}
        except Exception:
            return {}
    return {}


def save_bookmarks(aliases: Dict[str, str]) -> None:
    '''Persist project aliases to BOOKMARKS_PATH.'''
    BOOKMARKS_PATH.write_text(json.dumps({"aliases": aliases}, indent=2), encoding="utf-8")


## Next

In [ ]:
#| export

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()